[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/03-tabelas-tipos/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/03-tabelas-tipos")
    print("Material preparado em:", Path.cwd())


# Aula 03 — Tabelas e tipos de dados com Airbnb NYC

Material de estudo

# Objetivos

Este notebook usa a base **New York City Airbnb Open Data (2019)**,
publicada no Kaggle. Cada linha representa um anúncio e as colunas
descrevem localização, tipo de acomodação, preço, avaliações e
disponibilidade.

Ao final, você deverá conseguir:

- explicar o que representa uma linha da tabela;
- diferenciar tipo computacional e tipo semântico;
- inspecionar dimensões, colunas, ausências e duplicatas;
- converter categorias e datas;
- construir regras simples de validade;
- selecionar linhas e colunas com `loc`, `iloc` e `query`;
- criar colunas vetorizadas;
- resumir grupos com `groupby` e `agg`;
- documentar decisões de limpeza.

Fonte:
<https://www.kaggle.com/datasets/thedevastator/airbnbs-nyc-overview>.

## Como estudar este capítulo

Antes de analisar dados, precisamos entender o que a tabela representa.
Uma coluna chamada `price` pode parecer autoexplicativa, mas sua
interpretação depende da unidade, do período, da moeda e do processo de
coleta. Da mesma forma, o tipo `int64` informa como o computador
armazenou valores, não se a variável é uma medida, uma categoria ou um
identificador.

Este capítulo segue o percurso de uma primeira leitura profissional:
identificar a unidade observacional, consultar o dicionário, verificar
dimensões e tipos, investigar ausências e duplicatas, aplicar regras de
validade e somente então resumir grupos. Cada operação em pandas
responde a uma pergunta sobre a base; ela não deve ser executada apenas
porque faz parte de uma receita.

Antes de abrir cada código, formule o resultado esperado. Depois da
saída, descreva em palavras o que mudou na tabela, quantas observações
foram afetadas e que decisão analítica isso sustenta. O objetivo é
aprender simultaneamente a linguagem das tabelas e o raciocínio
necessário para trabalhar com elas.

# 1. Preparação

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

DATA = next(path for path in [
    Path("data/AB_NYC_2019.csv"),
    Path("exemplos/03-tabelas-tipos/data/AB_NYC_2019.csv"),
] if path.exists())

# 2. Leitura e unidade observacional

In [ ]:
airbnb = pd.read_csv(DATA)
airbnb.head(3)

**Unidade observacional:** um anúncio publicado na plataforma. A tabela
não registra diretamente reservas, hóspedes ou faturamento realizado.

In [ ]:
airbnb.shape

> **Interpretação**
>
> Antes de interpretar resultados, confirme o que cada linha representa,
> o período coberto e as colunas realmente disponíveis. Essa definição
> determina quais agregações e comparações são válidas.

In [ ]:
airbnb.columns.tolist()

## Dicionário das principais variáveis

| coluna | significado | tipo semântico |
|----|----|----|
| `id` | identificador do anúncio | identificador |
| `host_id` | identificador do anfitrião | identificador |
| `neighbourhood_group` | distrito | categoria nominal |
| `neighbourhood` | bairro | categoria nominal |
| `latitude`, `longitude` | posição geográfica | coordenadas |
| `room_type` | modalidade do anúncio | categoria nominal |
| `price` | preço anunciado por noite em US\$ | quantidade |
| `minimum_nights` | mínimo de noites | contagem |
| `number_of_reviews` | total de avaliações | contagem |
| `last_review` | data da avaliação mais recente | data |
| `reviews_per_month` | média mensal de avaliações | taxa |
| `availability_365` | dias disponíveis no ano | contagem limitada |

# 3. Diagnóstico inicial

In [ ]:
airbnb.info()

Observe que `last_review` chegou como `object`. O pandas conhece a
representação inicial, mas não conhece automaticamente todo o
significado da variável.

In [ ]:
diagnostico = pd.DataFrame({
    "dtype": airbnb.dtypes.astype(str),
    "ausentes": airbnb.isna().sum(),
    "unicos": airbnb.nunique(dropna=True),
})
diagnostico

## Pergunta de estudo 1

Por que `id` e `host_id`, embora armazenados como inteiros, não são
variáveis quantitativas? Tente explicar antes de abrir a resposta.

Resposta sugerida

Os números funcionam como rótulos. Somar, calcular a média ou
multiplicar IDs não produz uma quantidade interpretável.

# 4. Tipos mais adequados

In [ ]:
categoricas = ["neighbourhood_group", "neighbourhood", "room_type"]
airbnb[categoricas] = airbnb[categoricas].astype("category")

airbnb["last_review"] = pd.to_datetime(
    airbnb["last_review"], errors="coerce"
)

airbnb[categoricas + ["last_review"]].dtypes

## Datas permitem novas perguntas

In [ ]:
airbnb["review_year"] = airbnb["last_review"].dt.year.astype("Int64")
airbnb["review_month"] = airbnb["last_review"].dt.month.astype("Int64")

airbnb[["last_review", "review_year", "review_month"]].head()

# 5. Valores ausentes

In [ ]:
missing = airbnb.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing[missing > 0].sort_values().plot.barh(ax=ax, color="#0f6b78")
ax.set(title="Valores ausentes por coluna", xlabel="ausentes", ylabel="")
plt.show()

`last_review` e `reviews_per_month` faltam nas mesmas 10.052 linhas.
Vamos verificar se os padrões coincidem.

In [ ]:
mesmo_padrao = airbnb["last_review"].isna().equals(
    airbnb["reviews_per_month"].isna()
)
mesmo_padrao

In [ ]:
pd.crosstab(
    airbnb["number_of_reviews"].eq(0),
    airbnb["last_review"].isna(),
    rownames=["zero avaliações"],
    colnames=["data ausente"],
)

> **Interpretação**
>
> As ausências de `last_review` e `reviews_per_month` coincidem com
> anúncios sem avaliações, indicando um padrão estrutural: sem
> avaliação, não existem data da última avaliação nem média mensal.
> Ausência não equivale a zero; preencher a data ou a taxa com zero
> mudaria o significado dos dados.

# 6. Identificadores e duplicatas

In [ ]:
pd.Series({
    "linhas duplicadas": airbnb.duplicated().sum(),
    "IDs duplicados": airbnb["id"].duplicated().sum(),
    "ID é único": airbnb["id"].is_unique,
})

Anúncios com nomes iguais não são necessariamente duplicatas: pessoas
distintas podem usar títulos genéricos como “Cozy apartment”.

# 7. Regras de validade

In [ ]:
validacoes = pd.Series({
    "preço positivo": airbnb["price"].gt(0).mean(),
    "disponibilidade entre 0 e 365": airbnb["availability_365"].between(0, 365).mean(),
    "latitude plausível": airbnb["latitude"].between(40, 41).mean(),
    "longitude plausível": airbnb["longitude"].between(-75, -73).mean(),
})
validacoes

In [ ]:
airbnb.loc[airbnb["price"].eq(0), [
    "id", "name", "neighbourhood_group", "room_type", "price"
]].head()

Não vamos apagar automaticamente esses registros. Primeiro os marcamos
para que a decisão seja auditável.

In [ ]:
airbnb["price_valid"] = airbnb["price"].gt(0)
airbnb["price_extreme"] = airbnb["price"].gt(1000)
airbnb[["price_valid", "price_extreme"]].mean()

> **Interpretação**
>
> Todo filtro redefine a população analisada. Registre a condição, conte
> quantas linhas permanecem e explicite o denominador antes de
> interpretar proporções ou médias.

# 8. Distribuição do preço

In [ ]:
airbnb["price"].describe(percentiles=[.5, .9, .95, .99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(airbnb["price"], bins=60, ax=axes[0], color="#d95f02")
axes[0].set(title="Escala completa", xlabel="US$ por noite")

sns.histplot(
    airbnb.loc[airbnb["price"].between(1, 500), "price"],
    bins=50, ax=axes[1], color="#0f6b78"
)
axes[1].set(title="Zoom entre US$ 1 e 500", xlabel="US$ por noite")
plt.tight_layout()
plt.show()

O zoom deve ser declarado: ele melhora a leitura da região densa, mas
não pode ser usado para fingir que os valores extremos não existem.

# 9. Seleção de colunas e linhas

In [ ]:
airbnb["price"].head()  # Series

In [ ]:
airbnb[["neighbourhood_group", "room_type", "price"]].head()  # DataFrame

In [ ]:
airbnb.iloc[0:3, 0:5]

In [ ]:
airbnb.loc[0:2, ["id", "room_type", "price"]]

## Filtros legíveis

In [ ]:
recorte = airbnb.query(
    "neighbourhood_group == 'Manhattan' and "
    "room_type == 'Entire home/apt' and 0 < price <= 500"
).copy()

recorte.shape

Usamos `.copy()` porque criaremos novas colunas no recorte.

# 10. Operações vetorizadas

In [ ]:
recorte["price_brl_example"] = recorte["price"] * 5.0
recorte["potentially_unavailable_days"] = 365 - recorte["availability_365"]
recorte[["price", "price_brl_example", "availability_365",
         "potentially_unavailable_days"]].head()

A cotação de 5 reais por dólar é apenas didática. Em uma análise real,
registre a fonte e a data da taxa de câmbio.

# 11. Categorias e contagens

In [ ]:
airbnb["neighbourhood_group"].value_counts()

In [ ]:
airbnb["room_type"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
airbnb["neighbourhood_group"].value_counts().sort_values().plot.barh(
    ax=axes[0], color="#0f6b78"
)
airbnb["room_type"].value_counts().sort_values().plot.barh(
    ax=axes[1], color="#16826c"
)
axes[0].set(title="Anúncios por distrito", ylabel="")
axes[1].set(title="Anúncios por tipo", ylabel="")
plt.tight_layout()
plt.show()

# 12. Groupby: dividir, aplicar e combinar

In [ ]:
airbnb.groupby("room_type", observed=True)["price"].median()

In [ ]:
resumo_tipo = (
    airbnb
    .query("price > 0")
    .groupby("room_type", observed=True)
    .agg(
        anuncios=("id", "size"),
        preco_mediano=("price", "median"),
        preco_medio=("price", "mean"),
        disponibilidade_mediana=("availability_365", "median"),
    )
    .sort_values("preco_mediano", ascending=False)
)
resumo_tipo

Observe como média e mediana diferem. Os valores extremos puxam a média
para cima.

## Duas categorias

In [ ]:
medianas = (
    airbnb
    .query("price > 0")
    .groupby(["neighbourhood_group", "room_type"], observed=True)
    ["price"].median()
    .unstack()
)
medianas

In [ ]:
ax = medianas.plot.bar(figsize=(10, 5), color=["#0f6b78", "#d95f02", "#16826c"])
ax.set(title="Preço mediano por distrito e tipo", xlabel="", ylabel="US$ por noite")
ax.legend(title="tipo de acomodação", frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

> **Interpretação**
>
> Compare grupos usando a mesma definição de métrica e mostre também
> seus tamanhos. Diferenças aparentes podem refletir composição,
> cobertura ou poucos casos, e não apenas o fator usado no agrupamento.

# 13. Encadeamento documentado

In [ ]:
resultado = (
    airbnb
    .query("price > 0 and price <= 500")
    .assign(
        reviewed=lambda d: d["number_of_reviews"].gt(0),
        recent_review=lambda d: d["last_review"].ge("2019-01-01"),
    )
    .groupby(["neighbourhood_group", "room_type"], observed=True)
    .agg(
        n=("id", "size"),
        median_price=("price", "median"),
        reviewed_share=("reviewed", "mean"),
        recent_share=("recent_review", "mean"),
    )
    .reset_index()
    .sort_values(["median_price", "n"], ascending=[False, False])
)
resultado.head(10)

Cada etapa corresponde a uma decisão: filtrar preços, criar indicadores,
agrupar, agregar, recuperar colunas e ordenar.

# 14. Exercícios

## Exercício A — tipos

Classifique `minimum_nights`, `host_id`, `room_type`, `last_review` e
`reviews_per_month` em tipos computacionais e semânticos.

## Exercício B — validação

Crie indicadores para:

- `minimum_nights` entre 1 e 365;
- `reviews_per_month` não negativo quando presente;
- `availability_365` entre 0 e 365.

In [ ]:
# Escreva sua solução aqui.

## Exercício C — pergunta agrupada

Calcule, por distrito:

- número de anúncios;
- preço mediano entre US\$ 1 e 500;
- proporção de anúncios com pelo menos uma avaliação.

In [ ]:
# Escreva sua solução aqui.

# 15. Respostas sugeridas

In [ ]:
checks = airbnb.assign(
    valid_minimum_nights=airbnb["minimum_nights"].between(1, 365),
    valid_reviews_per_month=airbnb["reviews_per_month"].ge(0) |
                              airbnb["reviews_per_month"].isna(),
    valid_availability=airbnb["availability_365"].between(0, 365),
)

checks[["valid_minimum_nights", "valid_reviews_per_month",
        "valid_availability"]].mean()

In [ ]:
resposta_c = (
    airbnb
    .query("0 < price <= 500")
    .assign(reviewed=lambda d: d["number_of_reviews"].gt(0))
    .groupby("neighbourhood_group", observed=True)
    .agg(
        anuncios=("id", "size"),
        preco_mediano=("price", "median"),
        proporcao_com_review=("reviewed", "mean"),
    )
)
resposta_c

# 16. Checklist final

Antes de analisar uma tabela, registre:

1.  unidade observacional e população-alvo;
2.  origem e processo de coleta;
3.  significado, unidade e tipo de cada coluna;
4.  ausências, duplicatas e regras de validade;
5.  filtros e transformações aplicados;
6.  limitações que permanecem.

Uma análise reprodutível não é apenas código que executa: é uma
sequência de decisões que outra pessoa consegue compreender, verificar e
questionar.

# 17. Função reutilizável de auditoria

Uma boa prática é transformar verificações recorrentes em uma função.
Isso não substitui o conhecimento do domínio, mas reduz esquecimentos.

In [ ]:
def audit_table(df, id_column=None):
    """Produz um diagnóstico compacto de uma tabela."""
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
        "unique_n": df.nunique(dropna=True),
    })

    summary = {
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
    }

    if id_column is not None:
        summary["duplicate_ids"] = int(df[id_column].duplicated().sum())
        summary["missing_ids"] = int(df[id_column].isna().sum())

    return summary, report.sort_values("missing_pct", ascending=False)


summary, column_report = audit_table(airbnb, id_column="id")
summary

In [ ]:
column_report.head(10)

## Extensão

Modifique `audit_table` para receber um dicionário de intervalos
válidos, por exemplo
`{"availability_365": (0, 365), "price": (0, 10_000)}`, e contar
violações.

# 18. Tabelas cruzadas e proporções condicionais

Uma contagem simples pode esconder que grupos têm tamanhos muito
diferentes.

In [ ]:
counts = pd.crosstab(
    airbnb["neighbourhood_group"],
    airbnb["room_type"],
)
counts

Agora normalizamos cada linha para responder: “dentro de cada distrito,
qual é a composição por tipo de acomodação?”.

In [ ]:
row_percent = pd.crosstab(
    airbnb["neighbourhood_group"],
    airbnb["room_type"],
    normalize="index",
).mul(100).round(1)
row_percent

In [ ]:
ax = row_percent.plot.bar(
    stacked=True,
    figsize=(10, 4.8),
    color=["#0f6b78", "#d95f02", "#16826c"],
)
ax.set(title="Composição dos anúncios dentro de cada distrito",
       xlabel="", ylabel="percentual")
ax.legend(title="tipo", bbox_to_anchor=(1.02, 1), frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Cuidado com o denominador

- `normalize="index"`: percentual dentro de cada linha;
- `normalize="columns"`: percentual dentro de cada coluna;
- `normalize="all"`: percentual do total da tabela.

Sempre declare qual denominador gera a porcentagem.

# 19. Índices hierárquicos e reshape

Agrupar por duas colunas produz um índice com dois níveis.

In [ ]:
grouped = (
    airbnb
    .query("price > 0")
    .groupby(["neighbourhood_group", "room_type"], observed=True)
    ["price"].median()
)
grouped.head(8)

`unstack` move um nível do índice para as colunas.

In [ ]:
wide = grouped.unstack("room_type")
wide

`stack` faz o caminho inverso.

In [ ]:
wide.stack().head(8)

## Formato largo e formato longo

- **Largo:** uma coluna para cada tipo de acomodação; útil para leitura
  humana.
- **Longo:** uma linha para cada combinação; útil para gráficos e
  modelagem.

In [ ]:
long = wide.reset_index().melt(
    id_vars="neighbourhood_group",
    var_name="room_type",
    value_name="median_price",
)
long.head()

> **Interpretação**
>
> O formato longo facilita operações por grupo e gráficos com uma
> variável de categoria; o formato largo favorece comparações tabulares.
> A transformação muda a organização, não a informação substantiva.

# 20. Depuração de erros frequentes

## Erro 1 — usar `and` em Series

Este código falha:

``` python
airbnb[(airbnb["price"] > 0) and (airbnb["price"] <= 500)]
```

Use `&` e parênteses:

In [ ]:
airbnb[(airbnb["price"] > 0) & (airbnb["price"] <= 500)].shape

## Erro 2 — selecionar Series quando esperava DataFrame

In [ ]:
type(airbnb["price"]), type(airbnb[["price"]])

## Erro 3 — ordenar sem guardar o resultado

Muitos métodos retornam um novo objeto.

In [ ]:
sorted_prices = airbnb.sort_values("price", ascending=False)
sorted_prices[["id", "price"]].head()

## Erro 4 — converter datas silenciosamente

`errors="coerce"` transforma datas inválidas em `NaT`. Sempre conte
quantos valores foram perdidos durante a conversão.

In [ ]:
raw_dates = pd.read_csv(DATA, usecols=["last_review"])["last_review"]
parsed_dates = pd.to_datetime(raw_dates, errors="coerce")

pd.Series({
    "ausentes antes": raw_dates.isna().sum(),
    "ausentes depois": parsed_dates.isna().sum(),
})

# 21. Miniestudo guiado

## Pergunta

Entre anúncios com preço entre US\$ 1 e 500, como preço e
disponibilidade variam entre distritos e tipos de acomodação?

## Plano

1.  Definir a unidade: anúncio.
2.  Declarar o filtro de validade do preço.
3.  Descrever tamanhos dos grupos.
4.  Comparar medianas, não apenas médias.
5.  Visualizar a distribuição.
6.  Comunicar limitações.

In [ ]:
study = (
    airbnb
    .query("1 <= price <= 500")
    .dropna(subset=["neighbourhood_group", "room_type"])
    .copy()
)

study.shape

In [ ]:
study_summary = (
    study
    .groupby(["neighbourhood_group", "room_type"], observed=True)
    .agg(
        n=("id", "size"),
        median_price=("price", "median"),
        q1_price=("price", lambda x: x.quantile(.25)),
        q3_price=("price", lambda x: x.quantile(.75)),
        median_availability=("availability_365", "median"),
    )
    .reset_index()
)
study_summary.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(
    data=study,
    x="neighbourhood_group",
    y="price",
    hue="room_type",
    showfliers=False,
    ax=ax,
)
ax.set(title="Preço por distrito e tipo — outliers ocultados apenas na visualização",
       xlabel="", ylabel="US$ por noite")
ax.legend(title="tipo", frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Interpretação responsável

- Manhattan apresenta preços anunciados maiores em vários tipos.
- Apartamentos/casas inteiros tendem a custar mais que quartos privados.
- O gráfico descreve anúncios publicados, não preços efetivamente pagos.
- Ausência de ocupação e características do imóvel impede explicar
  causalmente as diferenças.

# 22. Desafios adicionais

1.  Encontre os cinco bairros com mais anúncios, mas exija pelo menos
    100 linhas.
2.  Compare média e mediana do preço em cada tipo de acomodação.
3.  Calcule a proporção de anúncios sem avaliação por distrito.
4.  Crie uma coluna `availability_band` com faixas `0`, `1–90`,
    `91–180`, `181–365` dias.
5.  Investigue anfitriões com muitos anúncios usando
    `calculated_host_listings_count`.
6.  Refaça o miniestudo usando somente anúncios com avaliação em 2019 e
    discuta como o filtro muda a população analisada.

# 23. Leituras adicionais

- [Pandas User Guide — 10 minutes to
  pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Pandas User Guide — Working with missing
  data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Pandas User Guide — Group
  by](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Tidy Data, Hadley
  Wickham](https://vita.had.co.nz/papers/tidy-data.pdf)
- [Kaggle — Airbnb in
  NYC](https://www.kaggle.com/datasets/thedevastator/airbnbs-nyc-overview)

# Guia teórico consolidado

## Dados são produzidos por um processo

Uma tabela não é a realidade: é o resultado de decisões sobre **quem
observar**, **o que medir**, **quando registrar** e **como codificar**.
Antes de calcular qualquer resumo, identifique:

- **unidade observacional:** o que uma linha representa;
- **população-alvo:** sobre quem desejamos concluir;
- **quadro amostral:** quem poderia efetivamente aparecer na coleta;
- **amostra:** quem de fato apareceu;
- **mecanismo de seleção:** como as unidades entraram na base.

Amostragem aleatória simples, estratificada, por conglomerados e
sistemática são desenhos probabilísticos distintos. Uma base
disponibilizada por uma plataforma, como o Airbnb, normalmente combina
seleção dos anfitriões, decisões comerciais e filtros administrativos;
ela não deve ser tratada automaticamente como amostra aleatória de todas
as acomodações da cidade.

## Tipo de armazenamento não é significado

O tipo técnico de uma coluna (`int`, `float`, `string`, `datetime`) não
determina seu papel analítico. Um CEP pode ser armazenado como número,
mas é um identificador; uma avaliação de 1 a 5 é numérica no arquivo,
mas possui interpretação ordinal.

- **categórico nominal:** categorias sem ordem natural;
- **categórico ordinal:** categorias com ordem, mas sem distâncias
  necessariamente iguais;
- **numérico discreto:** contagens;
- **numérico contínuo:** medidas em uma escala contínua;
- **identificador:** distingue unidades, mas não mede magnitude;
- **data/hora:** carrega ordem, duração, sazonalidade e fuso;
- **texto:** exige representação adequada ao objetivo.

Escolher o tipo semântico errado produz operações sem sentido, como
calcular a média de códigos postais.

## Ausência, duplicação e validade

Um valor ausente pode significar “não medido”, “não aplicável”,
“recusado” ou “perdido”. Substituí-lo por zero altera o significado. Da
mesma forma, duplicata depende da chave: duas linhas iguais podem ser
erro, mas duas linhas do mesmo hóspede podem representar reservas
diferentes.

Uma regra de validade combina domínio e contexto: preço deve ser não
negativo, latitude deve estar no intervalo geográfico plausível e
identificadores que deveriam ser únicos precisam ser testados como tal.

## Por que `groupby` funciona

O padrão **dividir–aplicar–combinar** separa a tabela em grupos, aplica
uma função a cada subconjunto e reúne os resultados. A unidade da tabela
resultante muda: depois de agrupar por bairro, uma linha passa a
representar um bairro, não um anúncio. Toda agregação perde detalhes;
por isso, visualize os dados individuais antes de resumir e declare a
nova unidade observacional.